In [1]:
import sympy as sp
from sympy import sqrt, eye, Matrix, binomial, S
from sympy.physics.quantum.trace import Tr
from sympy.physics.quantum.tensorproduct import TensorProduct
import dill

def amplitude_damping_kraus(dim, symbol):
    """
    Generate the Kraus operators for the amplitude damping channel
    """
    kraus_ops = []
    for k in range(dim):
        K = Matrix.zeros(dim, dim)
        for n in range(k, dim):
            amp = (
                sqrt(binomial(n, k))
                * ((1 - symbol) ** ((n - k) / S(2)))
                * (symbol ** (k / S(2)))
            )
            K[n - k, n] = amp
        kraus_ops.append(K)

    return kraus_ops

def get_SPDC_coeffs(lam):
    N0 = sp.sqrt(1 - lam**2)

    c0 = N0 * (lam**0)
    c1 = N0 * (lam**1)
    c2 = N0 * (lam**2)

    length = sp.sqrt(c0**2 + c1**2 + c2**2)

    return c0 / length, c1 / length, c2 / length

# Define the maximum number of photons in the system
N_max_photons = 6

# Define ion-excitation strength
q = sp.Symbol("q")

lam = sp.symbols("lambda")

p0, p1, p2 = get_SPDC_coeffs(lam)

# Define ion state vector
ket_0_ion = sp.Matrix([1, 0])
ket_1_ion = sp.Matrix([0, 1])

bra_0_ion = ket_0_ion.T
bra_1_ion = ket_1_ion.T

# Define photon state vector
ket_0_photon = sp.Matrix([1, 0, 0, 0, 0, 0, 0])
ket_1_photon = sp.Matrix([0, 1, 0, 0, 0, 0, 0])
ket_2_photon = sp.Matrix([0, 0, 1, 0, 0, 0, 0])
ket_3_photon = sp.Matrix([0, 0, 0, 1, 0, 0, 0])
ket_4_photon = sp.Matrix([0, 0, 0, 0, 1, 0, 0])
ket_5_photon = sp.Matrix([0, 0, 0, 0, 0, 1, 0])
ket_6_photon = sp.Matrix([0, 0, 0, 0, 0, 0, 1])

bra_0_photon = ket_0_photon.T
bra_1_photon = ket_1_photon.T
bra_2_photon = ket_2_photon.T
bra_3_photon = ket_3_photon.T
bra_4_photon = ket_4_photon.T
bra_5_photon = ket_5_photon.T
bra_6_photon = ket_6_photon.T

# Define identity operator
id_ion = eye(2)
id_photon = eye(7)
id_ion_all = eye(8)

# Define remote loss
p_r = sp.Symbol("p_r")

# Define local loss
p_l = sp.Symbol("p_l")

# Define Kraus operators for the remote loss
K_ops_remote = amplitude_damping_kraus(7, p_r)


# Define Kraus operators for the local loss
K_ops_memory = amplitude_damping_kraus(7, p_l)

# Define Kraus operators for the ion emission
K_ops_ion_emission = amplitude_damping_kraus(7, p_l)
ket_W = (1 / sqrt(3)) * (
    TensorProduct(ket_0_ion, ket_0_ion, ket_1_ion)
    + TensorProduct(ket_0_ion, ket_1_ion, ket_0_ion)
    + TensorProduct(ket_1_ion, ket_0_ion, ket_0_ion)
)
def normalize(dm):
    return dm / Tr(dm)
def remote_click(dm):
    # The input mode corresponds to detector mode |100>
    t_100 = (1 / sqrt(3)) * (
        1
        * TensorProduct(
            id_photon, ket_1_photon, id_photon, ket_0_photon, id_photon, ket_0_photon
        )
        + 1
        * TensorProduct(
            id_photon, ket_0_photon, id_photon, ket_1_photon, id_photon, ket_0_photon
        )
        + 1
        * TensorProduct(
            id_photon, ket_0_photon, id_photon, ket_0_photon, id_photon, ket_1_photon
        )
    )

    dm_100 = t_100.T @ dm @ t_100

    return dm_100
def loading_click_1(dm):
    # The input mode corresponds to detector mode |100>
    t_100 = (1 / sqrt(2)) * (
        TensorProduct(ket_1_photon, id_photon, id_photon, id_ion, ket_0_photon)
        + TensorProduct(ket_0_photon, id_photon, id_photon, id_ion, ket_1_photon)
    )

    dm_100 = t_100.T @ dm @ t_100

    return dm_100
def loading_click_2(dm):
    # The input mode corresponds to detector mode |100>
    t_100 = (1 / sqrt(2)) * (
        TensorProduct(ket_1_photon, id_photon, id_ion, id_ion, ket_0_photon)
        + TensorProduct(ket_0_photon, id_photon, id_ion, id_ion, ket_1_photon)
    )

    dm_100 = t_100.T @ dm @ t_100

    return dm_100
def loading_click_3(dm):
    # The input mode corresponds to detector mode |100>
    t_100 = (1 / sqrt(2)) * (
        TensorProduct(ket_1_photon, id_ion, id_ion, id_ion, ket_0_photon)
        + TensorProduct(ket_0_photon, id_ion, id_ion, id_ion, ket_1_photon)
    )

    dm_100 = t_100.T @ dm @ t_100

    return dm_100

# Vacuum manifold (lam^0)
sp_joint_spdc_ket = (p0 ** S(3)) * TensorProduct(
    ket_0_photon, ket_0_photon, ket_0_photon, ket_0_photon, ket_0_photon, ket_0_photon
)

# Two-photon manifold (lam^1)
sp_joint_spdc_ket += ((p0 ** S(2)) * (p1)) * (
    TensorProduct(
        ket_0_photon,
        ket_0_photon,
        ket_0_photon,
        ket_0_photon,
        ket_1_photon,
        ket_1_photon,
    )
    + TensorProduct(
        ket_0_photon,
        ket_0_photon,
        ket_1_photon,
        ket_1_photon,
        ket_0_photon,
        ket_0_photon,
    )
    + TensorProduct(
        ket_1_photon,
        ket_1_photon,
        ket_0_photon,
        ket_0_photon,
        ket_0_photon,
        ket_0_photon,
    )
)

# Four-photon manifold (lam^2)
sp_joint_spdc_ket += ((p0) * (p1 ** S(2))) * (
    TensorProduct(
        ket_1_photon,
        ket_1_photon,
        ket_1_photon,
        ket_1_photon,
        ket_0_photon,
        ket_0_photon,
    )
    + TensorProduct(
        ket_1_photon,
        ket_1_photon,
        ket_0_photon,
        ket_0_photon,
        ket_1_photon,
        ket_1_photon,
    )
    + TensorProduct(
        ket_0_photon,
        ket_0_photon,
        ket_1_photon,
        ket_1_photon,
        ket_1_photon,
        ket_1_photon,
    )
)

# Four-photon manifold (lam^2)
sp_joint_spdc_ket += ((p0 ** S(2)) * p2) * (
    TensorProduct(
        ket_0_photon,
        ket_0_photon,
        ket_0_photon,
        ket_0_photon,
        ket_2_photon,
        ket_2_photon,
    )
    + TensorProduct(
        ket_0_photon,
        ket_0_photon,
        ket_2_photon,
        ket_2_photon,
        ket_0_photon,
        ket_0_photon,
    )
    + TensorProduct(
        ket_2_photon,
        ket_2_photon,
        ket_0_photon,
        ket_0_photon,
        ket_0_photon,
        ket_0_photon,
    )
)


sp_joint_spdc_dm = normalize(sp_joint_spdc_ket @ sp_joint_spdc_ket.T)

joint_spdc_dm_damp1 = sp.zeros(sp_joint_spdc_dm.shape[0])
joint_spdc_dm_damp2 = sp.zeros(sp_joint_spdc_dm.shape[0])
joint_spdc_dm_damp3 = sp.zeros(sp_joint_spdc_dm.shape[0])

# Damp the first photon
for i in range(len(K_ops_remote)):
    joint_spdc_dm_damp1 += (
        TensorProduct(
            id_photon, K_ops_remote[i], id_photon, id_photon, id_photon, id_photon
        )
        @ sp_joint_spdc_dm
        @ TensorProduct(
            id_photon, K_ops_remote[i].T, id_photon, id_photon, id_photon, id_photon
        )
    )

# Damp the second photon
for i in range(len(K_ops_remote)):
    joint_spdc_dm_damp2 += (
        TensorProduct(
            id_photon, id_photon, id_photon, K_ops_remote[i], id_photon, id_photon
        )
        @ joint_spdc_dm_damp1
        @ TensorProduct(
            id_photon, id_photon, id_photon, K_ops_remote[i].T, id_photon, id_photon
        )
    )

# Damp the third photon
for i in range(len(K_ops_remote)):
    joint_spdc_dm_damp3 += (
        TensorProduct(
            id_photon, id_photon, id_photon, id_photon, id_photon, K_ops_remote[i]
        )
        @ joint_spdc_dm_damp2
        @ TensorProduct(
            id_photon, id_photon, id_photon, id_photon, id_photon, K_ops_remote[i].T
        )
    )

qm_click = remote_click(joint_spdc_dm_damp3)
prob_qm_click = sp.simplify(3 * Tr(qm_click))

qm_click = normalize(qm_click)
# ----- QM readout damping -----

qm_click_damp1 = sp.zeros(qm_click.shape[0])
qm_click_damp2 = sp.zeros(qm_click.shape[0])
qm_click_damp3 = sp.zeros(qm_click.shape[0])

# Damp first QM
for i in range(len(K_ops_memory)):
    qm_click_damp1 += (
        TensorProduct(K_ops_memory[i], id_photon, id_photon)
        @ qm_click
        @ TensorProduct(K_ops_memory[i].T, id_photon, id_photon)
    )

# Damp second QM
for i in range(len(K_ops_memory)):
    qm_click_damp2 += (
        TensorProduct(id_photon, K_ops_memory[i], id_photon)
        @ qm_click_damp1
        @ TensorProduct(id_photon, K_ops_memory[i].T, id_photon)
    )

# Damp third QM
for i in range(len(K_ops_memory)):
    qm_click_damp3 += (
        TensorProduct(id_photon, id_photon, K_ops_memory[i])
        @ qm_click_damp2
        @ TensorProduct(id_photon, id_photon, K_ops_memory[i].T)
    )

qm_click = qm_click_damp3
ion_photon_state = sqrt(1 - q) * TensorProduct(ket_1_ion, ket_0_photon) + sqrt(
    q
) * TensorProduct(ket_0_ion, ket_1_photon)
ion_photon_dm = ion_photon_state @ ion_photon_state.T
ion_photon_dm_damp = sp.zeros(ion_photon_dm.shape[0])
# ----- ion emission damping ---
for i in range(len(K_ops_ion_emission)):
    ion_photon_dm_damp += (
        TensorProduct(id_ion, K_ops_ion_emission[i])
        @ ion_photon_dm
        @ TensorProduct(id_ion, K_ops_ion_emission[i].T)
    )
qm_ion_1 = TensorProduct(qm_click, ion_photon_dm_damp)
qm_ion_1 = loading_click_1(qm_ion_1)

qm_ion_2 = TensorProduct(qm_ion_1, ion_photon_dm_damp)
qm_ion_2 = loading_click_2(qm_ion_2)

qm_ion_3 = TensorProduct(qm_ion_2, ion_photon_dm_damp)
qm_ion_3 = loading_click_3(qm_ion_3)

prob_load = sp.simplify(8 * Tr(qm_ion_3))
qm_ion_3 = normalize(qm_ion_3)
ion_dm = sp.simplify(qm_ion_3)

get_prob_qm_click = sp.lambdify([q, lam, p_r, p_l], prob_qm_click)
get_prob_load = sp.lambdify([q, lam, p_r, p_l], prob_load)
get_ion_dm = sp.lambdify([q, lam, p_r, p_l], ion_dm)

with open("hybrid_prob_click.dill", "wb") as file:
    dill.dump(prob_qm_click, file)

with open("hybrid_prob_load.dill", "wb") as file:
    dill.dump(prob_load, file)

with open("hybrid_ion_dm.dill", "wb") as file:
    dill.dump(ion_dm, file)